# Learning Objectives
In this notebook, you will learn Spark Dataframe APIs.

# Question List

Solve the following questions using Spark Dataframe APIs

### Join

1. easy - https://pgexercises.com/questions/joins/simplejoin.html
2. easy - https://pgexercises.com/questions/joins/simplejoin2.html
3. easy - https://pgexercises.com/questions/joins/self2.html 
4. medium - https://pgexercises.com/questions/joins/threejoin.html (three join)
5. medium - https://pgexercises.com/questions/joins/sub.html (subquery and join)

### Aggregation

1. easy - https://pgexercises.com/questions/aggregates/count3.html Group by order by
2. easy - https://pgexercises.com/questions/aggregates/fachours.html group by order by
3. easy - https://pgexercises.com/questions/aggregates/fachoursbymonth.html group by with condition 
4. easy - https://pgexercises.com/questions/aggregates/fachoursbymonth2.html group by multi col
5. easy - https://pgexercises.com/questions/aggregates/members1.html count distinct
6. med - https://pgexercises.com/questions/aggregates/nbooking.html group by multiple cols, join

### String & Date

1. easy - https://pgexercises.com/questions/string/concat.html format string
2. easy - https://pgexercises.com/questions/string/case.html WHERE + string function
3. easy - https://pgexercises.com/questions/string/reg.html WHERE + string function
4. easy - https://pgexercises.com/questions/string/substr.html group by, substr
5. easy - https://pgexercises.com/questions/date/series.html generate ts
6. easy - https://pgexercises.com/questions/date/bookingspermonth.html extract month from ts

### Author: Humza Inam

### Question

How can you produce a list of the start times for bookings by members named 'David Farrell'?

https://pgexercises.com/questions/joins/simplejoin.html

In [0]:
members_filtered = (
    spark.read.table("members")
    .filter("firstname = 'David' and surname = 'Farrell'")
    .select("memid")
)

df = spark.read.table("bookings").select("starttime", "memid").join(members_filtered, on="memid").select("starttime")

display(df)

### Question

How can you produce a list of the start times for bookings for tennis courts, for the date '2012-09-21'? Return a list of start time and facility name pairings, ordered by the time.

https://pgexercises.com/questions/joins/simplejoin2.html

In [0]:
from pyspark.sql.functions import col

facilities_filtered = (
    spark.read.table("facilities")
    .filter(col("name").contains("Tennis Court"))
    .select("facid", "name")
)

df = spark.read.table("bookings").select("starttime", "facid").filter(col("starttime").contains("2012-09-21")).join(facilities_filtered, on="facid").select("starttime", "name").orderBy("starttime")

display(df)

### Question

How can you output a list of all members, including the individual who recommended them (if any)? Ensure that results are ordered by (surname, firstname).

https://pgexercises.com/questions/joins/self2.html

In [0]:
from pyspark.sql.functions import col

members_rec = (
    spark.read.table("members")
    .select("firstname", "surname", "memid")
    .withColumnRenamed("firstname", "recfname")
    .withColumnRenamed("surname", "recsname")
)

df =  ( spark.read.table("members")
    .select("firstname", "surname", "recommendedBy")
    .join(members_rec, col("recommendedBy") == col("memid"),  how="left")
    .withColumnRenamed("firstname", "memfname")
    .withColumnRenamed("surname", "memsname")
    .select("memfname", "memsname", "recfname", "recsname")
    .orderBy(col("memsname"), col("memfname"))
)

display(df)

### Question

How can you produce a list of all members who have used a tennis court? Include in your output the name of the court, and the name of the member formatted as a single column. Ensure no duplicate data, and order by the member name followed by the facility name.

https://pgexercises.com/questions/joins/threejoin.html

In [0]:
from pyspark.sql.functions import col, concat_ws

facilities_filtered = (
    spark.read.table("facilities")
    .filter(col("name").contains("Tennis Court"))
    .select("facid", "name")
)

bookings_filtered = ( 
                     spark.read.table("bookings")
                     .select("memid", "facid")
                     .join(facilities_filtered, on="facid")
                     .select("memid", "name")
)

df = (
    spark.read.table("members")
    .select("firstname", "surname", "memid")
    .join(bookings_filtered, on="memid")
    .select(
        concat_ws(" ", col("firstname"), col("surname")).alias("member"),
        col("name").alias("facility")
        )
    .distinct()
    .orderBy("member", "facility")
)
display(df)

### Question

How can you output a list of all members, including the individual who recommended them (if any), without using any joins? Ensure that there are no duplicates in the list, and that each firstname + surname pairing is formatted as a column and ordered.

https://pgexercises.com/questions/joins/sub.html



In [0]:
from pyspark.sql.functions import col, concat_ws, udf
from pyspark.sql.types import StringType

members_formatted_recommender = (
    spark.read.table("members")
    .select(
        concat_ws(" ", col("firstname"), col("surname"))
        .alias("recommender"),
        col("memid").alias("recommender_id"))
)

recommender_map = dict(
    members_formatted_recommender
        .select("recommender_id", "recommender")
        .collect()
)

get_recommender = udf(
    lambda rid: recommender_map.get(rid),
    StringType()
)

df = (
    spark.read.table("members")
    .select(
        concat_ws(" ", col("firstname"), col("surname")).alias("member"),
        col("recommendedBy")
    )
    .withColumn("recommender", get_recommender(col("recommendedBy")))
    .select("member", "recommender")
    .distinct()
    .orderBy("member")
)

display(df)


### Question

Produce a count of the number of recommendations each member has made. Order by member ID.

https://pgexercises.com/questions/aggregates/count3.html

In [0]:
from pyspark.sql.functions import col

df = (
    spark.read.table("members")
    .filter(col("recommendedby").isNotNull())
    .groupBy("recommendedby")
    .count()
    .orderBy("recommendedby")
)

display(df)

### Question

Produce a list of the total number of slots booked per facility. For now, just produce an output table consisting of facility id and slots, sorted by facility id.

https://pgexercises.com/questions/aggregates/fachours.html

In [0]:
from pyspark.sql.functions import col, sum

df = (
    spark.read.table("bookings")
    .groupBy("facid")
    .agg(sum("slots").alias("Total Slots"))
    .orderBy("facid")
)

display(df)


### Question

Produce a list of the total number of slots booked per facility in the month of September 2012. Produce an output table consisting of facility id and slots, sorted by the number of slots.

https://pgexercises.com/questions/aggregates/fachoursbymonth.html


In [0]:
from pyspark.sql.functions import col, sum

df = (
    spark.read.table("bookings")
    .filter(
        (col("starttime") >= "2012-09-01") &
        (col("starttime") <= "2012-10-01")
    )
    .groupBy("facid")
    .agg(sum("slots").alias("Total Slots"))
    .orderBy(col("Total Slots"))
)

display(df)


### Question

Produce a list of the total number of slots booked per facility per month in the year of 2012. Produce an output table consisting of facility id and slots, sorted by the id and month.

https://pgexercises.com/questions/aggregates/fachoursbymonth2.html

In [0]:
from pyspark.sql.functions import col, year, month, sum

df = (
    spark.read.table("bookings")
    .filter(year(col("starttime")) == 2012)
    .groupBy(
        "facid",
        month(col("starttime")).alias("month")
    )
    .agg(sum("slots").alias("Total Slots"))
    .orderBy("facid", "month")
)

display(df)


### Question

Find the total number of members (including guests) who have made at least one booking.

https://pgexercises.com/questions/aggregates/members1.html

In [0]:
from pyspark.sql.functions import countDistinct

df = (
    spark.read.table("bookings")
    .agg(countDistinct("memid").alias("distinct_members"))
)

display(df)


### Question

Produce a list of each member name, id, and their first booking after September 1st 2012. Order by member ID.


https://pgexercises.com/questions/aggregates/nbooking.html

In [0]:
from pyspark.sql.functions import col, min

df = (
    spark.read.table("members").alias("m")
    .join(
        spark.read.table("bookings").alias("b"),
        col("m.memid") == col("b.memid"),
        "inner"
    )
    .filter(col("b.starttime") >= "2012-09-01")
    .groupBy(
        col("m.surname"),
        col("m.firstname"),
        col("m.memid")
    )
    .agg(min(col("b.starttime")).alias("starttime"))
    .orderBy(col("m.memid"))
)

display(df)


### Question

Output the names of all members, formatted as 'Surname, Firstname'

https://pgexercises.com/questions/string/concat.html

In [0]:
from pyspark.sql.functions import col, concat, lit

df = (
    spark.read.table("members")
    .select(concat(col("surname"), lit(", "), col("firstname")).alias("name"))
)

display(df)


### Question

Perform a case-insensitive search to find all facilities whose name begins with 'tennis'. Retrieve all columns.

https://pgexercises.com/questions/string/case.html

In [0]:
from pyspark.sql.functions import col, upper

df = (
    spark.read.table("facilities")
    .filter(upper(col("name")).like("TENNIS%"))
)

display(df)


### Question

You've noticed that the club's member table has telephone numbers with very inconsistent formatting. You'd like to find all the telephone numbers that contain parentheses, returning the member ID and telephone number sorted by member ID.

https://pgexercises.com/questions/string/reg.html


In [0]:
from pyspark.sql.functions import col

df = (
    spark.read.table("members")
    .select("memid", "telephone")
    .filter(col("telephone").rlike("[()]"))
)

display(df)


### Question

You'd like to produce a count of how many members you have whose surname starts with each letter of the alphabet. Sort by the letter, and don't worry about printing out a letter if the count is 0.

https://pgexercises.com/questions/string/substr.html


In [0]:
from pyspark.sql.functions import col, substring, count

df = (
    spark.read.table("members")
    .select(substring(col("surname"), 1, 1).alias("letter"))
    .groupBy("letter")
    .agg(count("letter").alias("count"))
    .orderBy("letter")
)

display(df)


### Question

Produce a list of all the dates in October 2012. They can be output as a timestamp (with time set to midnight) or a date.


https://pgexercises.com/questions/date/series.html

In [0]:
from pyspark.sql.functions import expr, explode, sequence, date_format

df = (
    spark.range(1)
    .select(
        explode(
            sequence(
                expr("timestamp '2012-10-01 00:00:00'"),
                expr("timestamp '2012-10-31 00:00:00'"),
                expr("interval 1 day")
            )
        ).alias("ts")
    )
    .select(date_format("ts", "yyyy-MM-dd HH:mm:ss").alias("ts"))
)

display(df)


### Question

Return a count of bookings for each month, sorted by month

https://pgexercises.com/questions/date/bookingspermonth.html

In [0]:
from pyspark.sql.functions import col, date_trunc, date_format, count

df = (
    spark.read.table("bookings")
    .withColumn("month", date_trunc("month", col("starttime")))
    .groupBy("month")
    .agg(count("starttime").alias("count"))
    .select(date_format("month", "yyyy-MM-dd HH:mm:ss").alias("month"), "count")
    .orderBy("month")
)

display(df)
